# ToolNode
ToolNode is a prebuilt node that executes tools in LangGraph workflows. It handles parallel tool execution, error handling, and state injection automatically.

## Tool with normal human-readable response
- The return value is converted to a ToolMessage.
- The model sees that text and decides what to do next.
- No agent state fields are changed unless the model or another tool does so later.
- Use this when the result is naturally human-readable text.

In [10]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from operator import add


# State Schema
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add]


# Define a simple tool
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    print(f"Adding {a} and {b}")
    return a + b


tool_list = [add_numbers]


# Input capture node
def capture_input_node(state: AgentState):
    """Capture the raw user input into the message history."""
    return {"messages": [state["messages"][-1]]}


# LLM Node
def chat_node(state: AgentState):
    """
    Chat node that processes the user request.
    """
    chat_model = ChatOllama(model="qwen3.5:4b")
    bound_chat_model = chat_model.bind_tools(tool_list)
    model_response = bound_chat_model.invoke(state["messages"])

    return {"messages": [model_response]}


# Create a graph
graph = StateGraph(AgentState)
graph.add_node("capture_input", capture_input_node)
graph.add_node("llm", chat_node)
graph.add_node("tools", ToolNode(tool_list))
graph.add_edge(START, "capture_input")
graph.add_edge("capture_input", "llm")
graph.add_conditional_edges("llm", tools_condition)
graph.add_edge("tools", "llm")
app = graph.compile()
user_input = "What is the sum of 5 and 7?"

initial_state = {
    "messages": [HumanMessage(content=user_input)],
}

for event in app.stream(initial_state):
    for node_name, update in event.items():
        print(f"\n[Node: {node_name}]")
        print(f"Response: {update['messages'][-1].content}\n{'-'*150}")


[Node: capture_input]
Response: What is the sum of 5 and 7?
------------------------------------------------------------------------------------------------------------------------------------------------------

[Node: llm]
Response: 
------------------------------------------------------------------------------------------------------------------------------------------------------
Adding 5 and 7

[Node: tools]
Response: 12
------------------------------------------------------------------------------------------------------------------------------------------------------

[Node: llm]
Response: The sum of 5 and 7 is 12.
------------------------------------------------------------------------------------------------------------------------------------------------------


## Tool with object response
- The object is serialized and sent back as tool output.
- The model can read specific fields and reason over them.
- Like string returns, this does not directly update graph state.
- Use this when downstream reasoning benefits from explicit fields instead of free-form text.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from operator import add


# State Schema
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    input: str


# Define a simple tool
@tool
def add_numbers(a: int, b: int) -> dict:
    """Get Structured output for adding two numbers."""
    return {"Num1": a, "Num2": b, "Sum": a + b}


tool_list = [add_numbers]


# Input capture node
def capture_input_node(state: AgentState):
    """Capture the raw user input into the message history."""
    return {"messages": [HumanMessage(content=state["input"])]}


# LLM Node
def chat_node(state: AgentState):
    """
    Chat node that processes the user request.
    """
    chat_model = ChatOllama(model="qwen3.5:4b")
    bound_chat_model = chat_model.bind_tools(tool_list)
    model_response = bound_chat_model.invoke(state["messages"])

    return {"messages": [model_response]}


# Create a graph
graph = StateGraph(AgentState)
graph.add_node("capture_input", capture_input_node)
graph.add_node("llm", chat_node)
graph.add_node("tools", ToolNode(tool_list))
graph.add_edge(START, "capture_input")
graph.add_edge("capture_input", "llm")
graph.add_conditional_edges("llm", tools_condition)
graph.add_edge("tools", "llm")
app = graph.compile()
user_input = "What is the sum of 5 and 7?"
initial_state = {"input": user_input}

for event in app.stream(initial_state, stream_mode="values"):
    for message in event["messages"]:
        if isinstance(message, HumanMessage):
            print(f"HumanMessage: {message.content}")
        elif isinstance(message, AIMessage):
            print(f"AIMessage: {message.content}")
        elif isinstance(message, ToolMessage):
            print(f"ToolMessage: {message.content}")
    print(f"\n{'-'*150}")


------------------------------------------------------------------------------------------------------------------------------------------------------
HumanMessage: What is the sum of 5 and 7?

------------------------------------------------------------------------------------------------------------------------------------------------------
HumanMessage: What is the sum of 5 and 7?
AIMessage: 

------------------------------------------------------------------------------------------------------------------------------------------------------
HumanMessage: What is the sum of 5 and 7?
AIMessage: 
ToolMessage: {"Num1": 5, "Num2": 7, "Sum": 12}

------------------------------------------------------------------------------------------------------------------------------------------------------
